<div align="center">
  <img src="https://raw.githubusercontent.com/NaumanHSA/neurosurfer/main/docs/assets/banner/neurosurfer_banner_white.png" alt="Neurosurfer" width="45%"/>
</div>

<br/>

# 03 — Graph Agents

This notebook introduces **Graph Agents** — neurosurfer's DAG-based multi-agent orchestration layer.

Instead of a single agent working in isolation, a **Graph** wires multiple agents, tools, and Python
functions into a directed acyclic graph. Each **node** is one unit of work; **edges** (`depends_on`)
declare data flow. The executor runs nodes in topological order, automatically parallelising
independent branches.

You'll learn:

1. **Graph primitives** — `Graph`, `GraphNode`, node kinds
2. **Python API** — build a graph from code
3. **YAML / dict** — the file-based equivalent
4. **GraphExecutor** — low-level runner; inspect per-node results
5. **React nodes** — tool-using agents as graph nodes
6. **WorkflowPackage** — the multi-file on-disk format
7. **WorkflowRunner** — production runner that wires tools automatically
8. **WorkflowRegistry** — save, list, and reload packages by name

**Model used:** `google/gemma-4-12b-qat` served via LM Studio on `localhost:1234`

---

## Contents
1. [Setup](#1-setup)
2. [What is a Graph](#2-what-is-a-graph)
3. [Building a Graph with Python](#3-building-a-graph-with-python)
4. [Running with GraphExecutor](#4-running-with-graphexecutor)
5. [Defining Graphs from YAML](#5-defining-graphs-from-yaml)
6. [React Nodes — Agents with Tools](#6-react-nodes--agents-with-tools)
7. [WorkflowPackage — Multi-file Format](#7-workflowpackage--multi-file-format)
8. [WorkflowRunner](#8-workflowrunner)
9. [WorkflowRegistry](#9-workflowregistry)

---

## 1. Setup

Same boilerplate as notebooks 01 and 02.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
from pathlib import Path

# Point Python at the repo root when running from tutorials/
repo_root = Path(os.getcwd()).parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import neurosurfer
print(f"neurosurfer {neurosurfer.__version__}")

In [ ]:
# ── LM Studio connection ──────────────────────────────────────────
# Make sure LM Studio is running with the model loaded and the local server ON.

LM_STUDIO_URL   = "http://localhost:1234/v1"
LM_STUDIO_MODEL = "google/gemma-4-12b-qat"   # adjust if your name differs
CONTEXT_WINDOW  = 32_768

from neurosurfer.llm.providers.openai import OpenAICompatProvider

provider = OpenAICompatProvider(
    model          = LM_STUDIO_MODEL,
    base_url       = LM_STUDIO_URL,
    api_key        = "lm-studio",
    context_window = CONTEXT_WINDOW,
)
print(f"Provider: {provider.model}")
print(f"Tool call style: {provider.capabilities.tool_call_style}")

In [ ]:
# ── JupyterIO ───────────────────────────────────────────────────
# Used by react nodes that need approval callbacks inside the graph.

class JupyterIO:
    """Auto-approving IOHandler for notebook demos."""

    async def ask(self, question: str, options=None) -> str:
        print(f"  [ask] {question}")
        return "yes"

    async def request_plan_approval(self, plan: str) -> tuple[bool, str]:
        print(f"  [plan approved]")
        return True, ""

    async def request_shell_approval(self, command: str, reason: str) -> bool:
        print(f"  [shell approved] {command}")
        return True

    async def request_write_approval(self, path: str, summary: str) -> str:
        print(f"  [write approved] {path}")
        return "once"

    def notify(self, message: str) -> None:
        print(f"  [notice] {message}")

print("JupyterIO ready.")

---

## 2. What is a Graph

A **Graph** is a Directed Acyclic Graph (DAG) where each **node** is a discrete unit of work
and **edges** (`depends_on`) define data flow. The executor walks the graph in topological order,
running independent nodes in the same layer in parallel.

```
  graph inputs
       │
       ▼
  [researcher]  ──────────────────────────┬
       │                                       │
       ▼                                       ▼
  [writer]                            [fact_checker]
       │                                       │
       └───────────► [publisher] ◄─────────┘
                          │
                          ▼
                       output
```

`researcher` runs first, then `writer` and `fact_checker` run in **parallel** (same layer),
then `publisher` runs once both complete.

### Node kinds

| Kind | Backed by | Typical use |
|---|---|---|
| `base` | Single LLM call (`OneShotAgent`) | Summarise, classify, draft — no tools |
| `react` | Full `AgenticLoop` / `ReactAgent` with tools | Multi-step reasoning, file access, web calls |
| `function` / `python` | Python callable (imported by path) | Deterministic transforms, preprocessing |
| `tool` | Neurosurfer `Tool.run()` directly | Structured data lookup, file write, HTTP |

### Key input field

Every LLM-backed node receives `user_intent` from the graph inputs as the primary task
description, plus the formatted outputs of any `depends_on` nodes as context.

---

## 3. Building a Graph with Python

Import `Graph` and `GraphNode` — both are Pydantic models, so validation runs at construction time.

In [ ]:
from neurosurfer.graph import Graph, GraphNode

# ── node 1: research ───────────────────────────────────────────────
researcher = GraphNode(
    id          = "researcher",
    kind        = "base",
    description = "Fact-finding node that condenses core concepts.",
    goal        = "Research the topic and produce exactly 5 key bullet points.",
)

# ── node 2: write ─────────────────────────────────────────────────
writer = GraphNode(
    id          = "writer",
    kind        = "base",
    description = "Writing node that turns bullet points into a polished explanation.",
    goal        = "Write a clear, 2-paragraph explanation from the research notes above.",
    depends_on  = ["researcher"],   # ← receives researcher's output as context
)

# ── graph ─────────────────────────────────────────────────────────
graph = Graph(
    name        = "content_pipeline",
    description = "Research a topic, then write a polished explanation.",
    nodes       = [researcher, writer],
    outputs     = ["writer"],   # ← whose output becomes graph.final["writer"]
)

print(f"Graph : {graph.name}")
print(f"Nodes : {[n.id for n in graph.nodes]}")
print(f"Outputs: {graph.outputs}")
for n in graph.nodes:
    deps = n.depends_on or []
    print(f"  {n.id:15s} kind={n.kind:8s} depends_on={deps}")

---

## 4. Running with GraphExecutor

`GraphExecutor` is the low-level DAG runner.  Pass a `Graph` and a `Provider`; call `.run(inputs)`
and get a `GraphExecutionResult` back.

**Inputs** are passed as a dict.  The `user_intent` key carries the primary task description that
every LLM node receives. Any additional keys are listed under *Additional inputs* in each node's
prompt.

In [ ]:
from neurosurfer.graph import GraphExecutor

executor = GraphExecutor(graph, provider=provider)

result = executor.run({"user_intent": "Explain how attention mechanisms work in Transformer models"})

print(result.execution_summary())
print(f"Succeeded : {result.succeeded}")
print(f"Errors    : {result.errors or 'none'}")

In [ ]:
# ── per-node results ────────────────────────────────────────────────
for node_id, nr in result.nodes.items():
    status = "ok" if nr.ok else ("skipped" if nr.skipped else "ERROR")
    print(f"\n── {node_id} [{status}] ({nr.duration_ms} ms) ──")
    if nr.ok:
        print(str(nr.raw_output)[:400])
    else:
        print(f"  {nr.error}")

# ── final outputs ──────────────────────────────────────────────────
print(f"\n{'─' * 60}")
print("Final output (writer node):")
print(result.final.get("writer", "(none)"))

### GraphExecutionResult fields

| Field / property | Type | What it contains |
|---|---|---|
| `result.nodes` | `dict[node_id, NodeExecutionResult]` | Every node's result |
| `result.final` | `dict[node_id, Any]` | Output of the declared `outputs` nodes |
| `result.errors` | `dict[node_id, str]` | Nodes that errored (empty = all ok) |
| `result.skipped` | `list[str]` | Nodes skipped because an upstream failed |
| `result.succeeded` | `bool` | True iff no errors |
| `result.execution_summary()` | `str` | One-line human summary |

Each `NodeExecutionResult` also carries `duration_ms`, `raw_output`, `error`, and `ok`.

---

## 5. Defining Graphs from YAML

The Python API and YAML format are isomorphic — every `GraphNode` field maps directly to a YAML key.
This matters because the **WorkflowPackage** format stores graphs as `graph.yaml` files on disk.

`load_graph_from_dict(data)` validates a raw dict (just as the Python constructor does).
`load_graph(path)` reads and parses a YAML or JSON file.

In [ ]:
# The Python graph from section 3 expressed as a plain dict (mirrors graph.yaml)
graph_dict = {
    "name": "content_pipeline",
    "description": "Research a topic, then write a polished explanation.",
    "nodes": [
        {
            "id": "researcher",
            "kind": "base",
            "goal": "Research the topic and produce exactly 5 key bullet points.",
        },
        {
            "id": "writer",
            "kind": "base",
            "goal": "Write a clear, 2-paragraph explanation from the research notes above.",
            "depends_on": ["researcher"],
        },
    ],
    "outputs": ["writer"],
}

from neurosurfer.graph import load_graph_from_dict

graph_from_dict = load_graph_from_dict(graph_dict)
print(f"Loaded: {graph_from_dict.name} — {len(graph_from_dict.nodes)} nodes")

In [ ]:
# Write it to a YAML file and load it back via load_graph(path)
import yaml
from neurosurfer.graph.engine.loader import load_graph

tmp_yaml = Path("/tmp/tutorial_content_pipeline.yaml")
tmp_yaml.write_text(yaml.dump(graph_dict), encoding="utf-8")
print("Written to:", tmp_yaml)
print("YAML contents:")
print(tmp_yaml.read_text())

graph_from_file = load_graph(tmp_yaml)
print(f"Loaded from file: {graph_from_file.name} — {len(graph_from_file.nodes)} nodes")

---

## 6. React Nodes — Agents with Tools

A `react` node runs a full **`AgenticLoop`** (or `ReactAgent` for models without native tool calling)
with access to a subset of the neurosurfer tool pool. Declare the tool names in the node's `tools`
list — the executor selects only those tools for that node.

When using `GraphExecutor` directly you must supply a `ToolPool` and a `ToolContext`.  The higher-
level `WorkflowRunner` (section 8) builds the pool for you automatically from the declared tool names.

In [ ]:
from neurosurfer.graph import Graph, GraphNode, GraphExecutor
from neurosurfer.tools import default_pool
from neurosurfer.tools.base import ToolContext

# ── graph with a react node followed by a base node ──────────────────
react_graph = Graph(
    name="file_reporter",
    nodes=[
        GraphNode(
            id          = "scout",
            kind        = "react",
            description = "Explores the project and gathers file information.",
            goal        = (
                "Use list_dir to explore the project directory. "
                "Read the README.md and one other key file. "
                "Summarise what you find in bullet points, then call finish."
            ),
            tools       = ["list_dir", "read_file", "finish"],
        ),
        GraphNode(
            id          = "reporter",
            kind        = "base",
            description = "Turns the scout's findings into a short report.",
            goal        = "Write a 3-5 sentence project summary from the information gathered above.",
            depends_on  = ["scout"],
        ),
    ],
    outputs=["reporter"],
)

# ── supply native_tools + tool_ctx to the executor ──────────────────
tool_pool = default_pool().select(["list_dir", "read_file", "finish"])
tool_ctx  = ToolContext(cwd=repo_root, io=JupyterIO())

executor = GraphExecutor(
    react_graph,
    provider     = provider,
    native_tools = tool_pool,
    tool_ctx     = tool_ctx,
)

result = executor.run({"user_intent": "Summarise this Python project by reading its key files."})
print(result.execution_summary())

In [ ]:
print("Scout (react node) output:")
print(result.nodes["scout"].raw_output)

print(f"\n{'─' * 60}")
print("Reporter (base node) output:")
print(result.final.get("reporter", "(none)"))

---

## 7. WorkflowPackage — Multi-file Format

For production use, graphs live on disk as a **WorkflowPackage** directory:

```
my_workflow/
  workflow.yaml          ← manifest: name, version, description, entrypoint
  graph.yaml             ← graph spec (nodes, edges, outputs)
  agents/<id>.yaml       ← optional per-node overrides merged at load time
  nodes/<id>.py          ← Python callables for function/python nodes
  schemas.py             ← Pydantic output-schema classes
```

Only `workflow.yaml` and `graph.yaml` are required. `load_package(path)` loads the directory into
an in-memory `WorkflowPackage`.

In [ ]:
import yaml
from pathlib import Path

# ── create the package directory on disk ────────────────────────────
pkg_dir = Path("/tmp/content_pipeline")
pkg_dir.mkdir(parents=True, exist_ok=True)

# workflow.yaml — the manifest
(pkg_dir / "workflow.yaml").write_text(
    yaml.dump({
        "name": "content_pipeline",
        "version": "1.0.0",
        "description": "Research a topic and write a polished explanation.",
        "entrypoint": "graph.yaml",
    }),
    encoding="utf-8",
)

# graph.yaml — the graph spec
(pkg_dir / "graph.yaml").write_text(
    yaml.dump({
        "name": "content_pipeline",
        "nodes": [
            {
                "id": "researcher",
                "kind": "base",
                "goal": "Research the topic and produce exactly 5 key bullet points.",
            },
            {
                "id": "writer",
                "kind": "base",
                "goal": "Write a clear, 2-paragraph explanation from the research notes above.",
                "depends_on": ["researcher"],
            },
        ],
        "outputs": ["writer"],
    }),
    encoding="utf-8",
)

print("Package directory:", pkg_dir)
for f in sorted(pkg_dir.rglob("*")):
    print(f"  {f.relative_to(pkg_dir)}")

In [ ]:
from neurosurfer.graph.workflow import load_package

pkg = load_package(pkg_dir)

print(f"Name       : {pkg.name}")
print(f"Version    : {pkg.version}")
print(f"Description: {pkg.description}")
print(f"Graph nodes: {[n.id for n in pkg.graph.nodes]}")
print(f"Outputs    : {pkg.graph.outputs}")

---

## 8. WorkflowRunner

`WorkflowRunner` is the **recommended** way to execute a `WorkflowPackage`. It:

- Builds a `ToolPool` automatically from the tool names declared in the graph nodes
- Patches `sys.path` so function-node callables in `nodes/` can be imported
- Validates required graph inputs before running
- Accepts an optional **progress callback** `(node_id, status, duration_ms)` for live UI updates

In [ ]:
from neurosurfer.graph.workflow import WorkflowRunner

runner = WorkflowRunner(
    provider,
    cwd=repo_root,   # working directory for tool contexts
)

result = runner.run(
    pkg,
    inputs={"user_intent": "Explain gradient descent in machine learning"},
)

print(result.execution_summary())
print(f"\n{'─' * 60}")
print(result.final.get("writer", "(none)"))

In [ ]:
# ── progress callback ────────────────────────────────────────────────
# Fired once per node after it completes. Useful for live tables / progress bars.

def on_progress(node_id: str, status: str, duration_ms: int) -> None:
    icon = "\u2713" if status == "ok" else "\u2717"
    print(f"  {icon} {node_id:20s} [{status}]  {duration_ms} ms")

print("Running with progress callback\u2026")
result2 = runner.run(
    pkg,
    inputs    = {"user_intent": "Explain backpropagation in neural networks"},
    progress  = on_progress,
)
print(f"\nDone — {result2.execution_summary()}")

---

## 9. WorkflowRegistry

`WorkflowRegistry` is a **persistent store** for workflow packages. It saves each package to
`<registry_dir>/<name>/` and lets you list, retrieve, and delete them by name.

The default registry lives at `.neurosurfer/workflows/` in the current directory (configurable
via `NEUROSURFER_HOME`). Pass an explicit `workflows_dir` for tests or isolated environments.

In [ ]:
from neurosurfer.graph.workflow import WorkflowRegistry

# Use a temp dir so the demo doesn't pollute the project registry
registry = WorkflowRegistry(workflows_dir=Path("/tmp/ns_tutorial_registry"))

# ── save the package we created earlier ─────────────────────────────
registry.save(pkg)
print("Saved:", registry.list())

In [ ]:
# ── retrieve and run from the registry ─────────────────────────────
loaded_pkg = registry.get("content_pipeline")

print(f"Retrieved: {loaded_pkg.name} v{loaded_pkg.version}")
print(f"Nodes    : {[n.id for n in loaded_pkg.graph.nodes]}")

# Run it exactly like a freshly-loaded package
result3 = WorkflowRunner(provider, cwd=repo_root).run(
    loaded_pkg,
    inputs={"user_intent": "Explain the difference between RNNs and Transformers"},
)

print(f"\n{result3.execution_summary()}")
print(f"\n{'─' * 60}")
print(result3.final.get("writer", "(none)"))

In [ ]:
# ── registry management ─────────────────────────────────────────────
print("Available workflows:", registry.list())
print("Exists 'content_pipeline':", registry.exists("content_pipeline"))
print("Exists 'missing'          :", registry.exists("missing"))

# Delete it and confirm it's gone
registry.delete("content_pipeline")
print("After delete:", registry.list())

---

## Summary

| Concept | API | Typical use |
|---|---|---|
| Define a graph in code | `Graph(name=..., nodes=[GraphNode(...), ...])` | Prototyping, testing |
| Define a graph in YAML | `graph.yaml` loaded by `load_graph(path)` | Production, version-controlled |
| Run a graph directly | `GraphExecutor(graph, provider=...).run(inputs)` | Low-level, full control |
| React node with tools | `GraphNode(kind="react", tools=[...])` + `ToolPool` | Multi-step reasoning in a graph |
| Multi-file package | `workflow.yaml` + `graph.yaml` directory | Shareable, on-disk packages |
| Load a package | `load_package(path)` | Any package directory |
| Run a package | `WorkflowRunner(provider).run(pkg, inputs)` | Recommended production path |
| Persist packages | `WorkflowRegistry(dir).save(pkg)` | Save, list, retrieve by name |

**Key ideas**
- `user_intent` is the primary graph input key — every LLM node sees it as its task.
- Nodes in the same topological layer run in **parallel** automatically.
- `depends_on` outputs are injected into the downstream node's context verbatim.
- `WorkflowRunner` builds the `ToolPool` from declared tool names so you don't wire it by hand.
- The registry stores packages as directories — inspect or version-control them directly.

---

## What's Next

| # | Notebook | Topic |
|---|---|---|
| 01 | *Providers, Agents & RAG* | Core building blocks |
| 02 | *Custom Tools* | Define and register your own tools |
| **03** | *Graph Agents* ← **you are here** | Multi-agent DAG workflows |
| **04** | MCP Servers | Connect external MCP servers; (soon) host workflows |
| 05 | The Gateway Server | Serve agents as OpenAI-compatible endpoints |